In [ ]:
import os

radar_nc_dir = "radar_data"

# List files
files = sorted(os.listdir(radar_nc_dir))

print("Number of files found:", len(files))
print("First 5 files:")
for f in files[:5]:
    print("  ", f)


In [ ]:
import os
import numpy as np

base_dir = "/nfs/tcdynasty/esmith/RADAR/radar_data"
prefix = "20140914I1radar"

ctl_file = os.path.join(base_dir, prefix + ".ctl")
dat_file = os.path.join(base_dir, prefix + ".dat")

print("CTL exists:", os.path.exists(ctl_file))
print("DAT exists:", os.path.exists(dat_file))


In [ ]:
def read_ctl(ctl_path):
    meta = {}
    with open(ctl_path) as f:
        for line in f:
            if line.startswith("XDEF"):
                _, nx, _, x0, dx = line.split()
                meta["nx"] = int(nx)
                meta["x0"] = float(x0)
                meta["dx"] = float(dx)

            if line.startswith("YDEF"):
                _, ny, _, y0, dy = line.split()
                meta["ny"] = int(ny)
                meta["y0"] = float(y0)
                meta["dy"] = float(dy)

            if line.startswith("ZDEF"):
                _, nz, *_ = line.split()
                meta["nz"] = int(nz)

    return meta


In [ ]:
meta = read_ctl(ctl_file)
print(meta)


In [ ]:
with open(ctl_file) as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i > 25:
            break


In [ ]:
def read_ctl(ctl_path):
    meta = {}
    with open(ctl_path) as f:
        for line in f:
            parts = line.strip().split()

            if not parts:
                continue

            key = parts[0].upper()

            if key == "XDEF":
                meta["nx"] = int(parts[1])
                meta["x0"] = float(parts[3])
                meta["dx"] = float(parts[4])

            elif key == "YDEF":
                meta["ny"] = int(parts[1])
                meta["y0"] = float(parts[3])
                meta["dy"] = float(parts[4])

            elif key == "ZDEF":
                meta["nz"] = int(parts[1])

            elif key == "VARS":
                meta["nvars"] = int(parts[1])

    return meta


In [ ]:
meta = read_ctl(ctl_file)
print(meta)


In [ ]:
raw = np.fromfile(dat_file, dtype=">f4")

expected = meta["nvars"] * meta["nz"] * meta["ny"] * meta["nx"]

print("Raw data size:", raw.size)
print("Expected size:", expected)


In [ ]:
expected = meta["nvars"] * meta["nz"] * meta["ny"] * meta["nx"]

raw = raw[:expected]


In [ ]:
data = raw.reshape(
    meta["nvars"],
    meta["nz"],
    meta["ny"],
    meta["nx"]
)

print("Reshaped data shape:", data.shape)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Extract reflectivity (var index 3, lowest vertical level)
Z0 = data[3, 0, :, :].astype(float)

# Mask bad / non-physical values
Z0 = np.ma.masked_invalid(Z0)
Z0 = np.ma.masked_where((Z0 < -10.0) | (Z0 > 80.0), Z0)

# Plot
plt.figure(figsize=(6, 5))
im = plt.imshow(
    Z0,
    origin="lower",
    cmap="turbo",
    vmin=0,
    vmax=50
)
plt.colorbar(im, label="Reflectivity (dBZ)")
plt.title("Reflectivity (lowest level)")
plt.tight_layout()
plt.show()


In [ ]:
print("Valid points:", Z0.count())
print("Min (masked):", Z0.min())
print("Max (masked):", Z0.max())


In [ ]:
def parse_ctl(ctl_path):
    meta = {}
    with open(ctl_path, "r") as f:
        for line in f:
            if line.startswith("XDEF"):
                meta["nx"] = int(line.split()[1])
            elif line.startswith("YDEF"):
                meta["ny"] = int(line.split()[1])
            elif line.startswith("ZDEF"):
                meta["nz"] = int(line.split()[1])
            elif line.startswith("UNDEF"):
                meta["undef"] = float(line.split()[1])
            elif line.startswith("VARS"):
                meta["nv"] = int(line.split()[1])
    return meta


In [ ]:
with open(dat_file, "rb") as f:
    print("big endian:",  np.fromfile(f, dtype=">i4", count=1)[0])
    f.seek(0)
    print("little endian:", np.fromfile(f, dtype="<i4", count=1)[0])


In [ ]:
def read_grads_sequential(fname, nx, ny, nz, nv):
    nxy = nx * ny
    data = np.empty((nv, nz, ny, nx), dtype=np.float32)

    with open(fname, "rb") as f:
        for v in range(nv):
            for z in range(nz):
                # leading record marker (little endian)
                reclen = np.fromfile(f, dtype="<i4", count=1)[0]

                # read slab
                slab = np.fromfile(f, dtype="<f4", count=nxy)
                data[v, z, :, :] = slab.reshape(ny, nx)

                # trailing record marker
                _ = np.fromfile(f, dtype="<i4", count=1)[0]

    return data


In [ ]:
data = read_grads_sequential(
    dat_file,
    nx=105,
    ny=103,
    nz=37,
    nv=7
)

Z0 = data[3, 0]  # Reflectivity, lowest level

print("Z min:", np.nanmin(Z0))
print("Z max:", np.nanmax(Z0))


In [ ]:
Z0 = data[3, 0]  # Reflectivity, lowest level

Z0 = np.ma.masked_equal(Z0, -9999)

print("Valid min:", Z0.min())
print("Valid max:", Z0.max())


In [ ]:
plt.figure(figsize=(6,5))
plt.imshow(
    Z0,
    origin="lower",
    cmap="turbo",
    vmin=0,
    vmax=50
)
plt.colorbar(label="Reflectivity (dBZ)")
plt.title("Radar Reflectivity (lowest level)")
plt.show()


In [ ]:
from datetime import datetime

def get_time_from_ctl(ctl_path):
    with open(ctl_path, "r") as f:
        for line in f:
            if line.strip().startswith("TDEF"):
                tstr = line.split()[3]   # e.g. 15:00Z14SEP2014
                return datetime.strptime(tstr, "%H:%MZ%d%b%Y")
    return None


In [ ]:
# Get time from ctl file
time = get_time_from_ctl(ctl_file)

# Create full title
title = "Radar Reflectivity (lowest level)"
if time is not None:
    title += f"\n{time.strftime('%Y-%m-%d %H:%M UTC')}"

# Plot
plt.figure(figsize=(6,5))
plt.imshow(
    Z0,
    origin="lower",
    cmap="turbo",
    vmin=0,
    vmax=50
)
plt.colorbar(label="Reflectivity (dBZ)")
plt.title(title)
plt.show()


In [ ]:
# ---------------------------------- CODE FOR THE LOOP   ----------------------------------------------

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from datetime import datetime

# --- Functions ---
def get_time_from_ctl(ctl_path):
    """Extract start time from a .ctl file."""
    with open(ctl_path, "r") as f:
        for line in f:
            if line.strip().startswith("TDEF"):
                tstr = line.split()[3]  # e.g., 15:00Z14SEP2014
                return datetime.strptime(tstr, "%H:%MZ%d%b%Y")
    return None

# --- Paths ---
radar_dir = "/nfs/tcdynasty/esmith/RADAR/radar_data"
ctl_file = os.path.join(radar_dir, "20140914I1radar.ctl")
dat_file = os.path.join(radar_dir, "20140914I1radar.dat")

# --- Metadata ---
nx, ny, nz = 105, 103, 37  # XDEF, YDEF, ZDEF
nv = 7  # number of variables
R_index = 3  # Reflectivity index
undef_val = -9999

# --- Read raw data ---
raw = np.fromfile(dat_file, dtype=">f4")

expected_size = nv * nz * ny * nx
if raw.size > expected_size:
    raw = raw[:expected_size]

data = raw.reshape((nv, nz, ny, nx))

# --- Extract time ---
time = get_time_from_ctl(ctl_file)
time_str = time.strftime("%Y-%m-%d %H:%M UTC") if time else "Unknown time"

# --- Set up figure ---
fig, ax = plt.subplots(figsize=(6,5))

# Initial frame
Z0 = data[R_index, 0, :, :]
Z0 = np.ma.masked_where(
    (Z0 < -10) | (Z0 > 80) | (Z0 == undef_val) | np.isnan(Z0),
    Z0
)
im = ax.imshow(Z0, origin="lower", cmap="turbo", vmin=0, vmax=50)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Reflectivity (dBZ)")
title = ax.set_title(f"Radar Reflectivity (lowest level)\n{time_str}")

# --- Update function ---
def update(frame):
    Z0 = data[R_index, frame, :, :]
    Z0 = np.ma.masked_where(
        (Z0 < -10) | (Z0 > 80) | (Z0 == undef_val) | np.isnan(Z0),
        Z0
    )
    im.set_data(Z0)
    title.set_text(f"Radar Reflectivity (level 0)\n{time_str} - Frame {frame+1}")
    return [im, title]

# --- Animate ---
num_frames = nz  # using vertical levels as time steps here
anim = FuncAnimation(fig, update, frames=num_frames, interval=500, blit=True)

# --- Save as MP4 ---
writer = FFMpegWriter(fps=2)
anim.save("radar_animation_single_flight.mp4", writer=writer)
plt.close(fig)
print("Animation saved as radar_animation_single_flight.mp4")


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from datetime import datetime

# --- Functions ---
def get_time_from_ctl(ctl_path):
    """Extract start time from a .ctl file."""
    with open(ctl_path, "r") as f:
        for line in f:
            if line.strip().startswith("TDEF"):
                tstr = line.split()[3]  # e.g., 15:00Z14SEP2014
                return datetime.strptime(tstr, "%H:%MZ%d%b%Y")
    return None

# --- Paths ---
radar_dir = "/nfs/tcdynasty/esmith/RADAR/radar_data"
ctl_file = os.path.join(radar_dir, "20140914I1radar.ctl")
dat_file = os.path.join(radar_dir, "20140914I1radar.dat")

# --- Metadata ---
nx, ny, nz = 105, 103, 37  # XDEF, YDEF, ZDEF
nv = 7  # number of variables
R_index = 3  # Reflectivity
undef_val = -9999
lowest_levels_to_avg = 5  # average lowest N levels for smoother image

# --- Read data (GRADS sequential) ---
data = np.empty((nv, nz, ny, nx), dtype=np.float32)
with open(dat_file, "rb") as f:
    for v in range(nv):
        for z in range(nz):
            # leading record marker
            _ = np.fromfile(f, dtype="<i4", count=1)
            slab = np.fromfile(f, dtype="<f4", count=ny*nx)
            data[v, z, :, :] = slab.reshape(ny, nx)
            # trailing record marker
            _ = np.fromfile(f, dtype="<i4", count=1)

# --- Average lowest few levels ---
Z0 = np.mean(data[R_index, :lowest_levels_to_avg, :, :], axis=0)
Z0 = np.ma.masked_where((Z0 < 0) | (Z0 == undef_val) | np.isnan(Z0), Z0)

# --- Extract time ---
time = get_time_from_ctl(ctl_file)
time_str = time.strftime("%Y-%m-%d %H:%M UTC") if time else "Unknown time"

# --- Plot static reflectivity ---
plt.figure(figsize=(6,5))
plt.imshow(Z0, origin="lower", cmap="turbo", vmin=0, vmax=50)
plt.colorbar(label="Reflectivity (dBZ)")
plt.title(f"Radar Reflectivity (lowest {lowest_levels_to_avg} levels)\n{time_str}")
plt.show()


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from datetime import datetime, timedelta

# --- Functions ---
def get_time_from_ctl(ctl_path):
    """Extract start time and interval from a .ctl file."""
    start_time = None
    interval = None
    with open(ctl_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if line.startswith("TDEF"):
                # Example: TDEF 1 LINEAR 15:00Z14SEP2014 6hr
                start_time_str = parts[3]
                interval_str = parts[4]
                start_time = datetime.strptime(start_time_str, "%H:%MZ%d%b%Y")
                if interval_str.endswith("hr"):
                    interval = int(interval_str[:-2])
    return start_time, interval

# --- Paths ---
radar_dir = "/nfs/tcdynasty/esmith/RADAR/radar_data"
ctl_file = os.path.join(radar_dir, "20140914I1radar.ctl")
dat_file = os.path.join(radar_dir, "20140914I1radar.dat")

# --- Metadata ---
nx, ny, nz = 105, 103, 37
nv = 7
R_index = 3
undef_val = -9999
lowest_levels_to_avg = 5

# --- Read GRADS sequential file ---
data = np.empty((nv, nz, ny, nx), dtype=np.float32)
with open(dat_file, "rb") as f:
    for v in range(nv):
        for z in range(nz):
            _ = np.fromfile(f, dtype="<i4", count=1)
            slab = np.fromfile(f, dtype="<f4", count=ny*nx)
            data[v, z, :, :] = slab.reshape(ny, nx)
            _ = np.fromfile(f, dtype="<i4", count=1)

# --- Time info ---
start_time, interval = get_time_from_ctl(ctl_file)
if start_time is None:
    start_time = datetime(2014, 9, 14, 15, 0)  # fallback to known Edouard time
if interval is None:
    interval = 1

# --- Set up figure ---
fig, ax = plt.subplots(figsize=(6,5))
Z0 = np.mean(data[R_index, :lowest_levels_to_avg, :, :], axis=0)
Z0 = np.ma.masked_where((Z0 < 0) | (Z0 == undef_val) | np.isnan(Z0), Z0)
im = ax.imshow(Z0, origin="lower", cmap="turbo", vmin=0, vmax=50)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Reflectivity (dBZ)")
title = ax.set_title("Radar Reflectivity (lowest levels)")

# --- Update function ---
def update(frame):
    # For now, loop through vertical levels as pseudo-time
    Z0 = data[R_index, frame % nz, :, :]
    Z0 = np.ma.masked_where((Z0 < 0) | (Z0 == undef_val) | np.isnan(Z0), Z0)
    im.set_data(Z0)
    
    t = start_time + timedelta(hours=frame*interval)
    title.set_text(f"Radar Reflectivity (level {frame % nz})\n{t.strftime('%Y-%m-%d %H:%M UTC')}")
    return [im, title]

# --- Animate ---
frames = nz  # one frame per vertical level
anim = FuncAnimation(fig, update, frames=frames, interval=500, blit=True)

# --- Save animation ---
writer = FFMpegWriter(fps=2)
anim.save("radar_animation_single_flight.mp4", writer=writer)
plt.close(fig)
print("Animation saved as radar_animation_single_flight.mp4")


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from datetime import datetime

# --- Functions ---
def get_time_from_ctl(ctl_path):
    """Extract start time from a .ctl file."""
    with open(ctl_path, "r") as f:
        for line in f:
            if line.strip().startswith("TDEF"):
                tstr = line.split()[3]  # e.g., 15:00Z14SEP2014
                return datetime.strptime(tstr, "%H:%MZ%d%b%Y")
    return None

# --- Paths ---
radar_dir = "/nfs/tcdynasty/esmith/RADAR/radar_data"
ctl_file = os.path.join(radar_dir, "20140914I1radar.ctl")
dat_file = os.path.join(radar_dir, "20140914I1radar.dat")

# --- Metadata (from ctl file) ---
nx, ny, nz = 105, 103, 37  # XDEF, YDEF, ZDEF
nv = 7                       # number of variables
R_index = 3                  # Reflectivity index
undef_val = -9999

# XDEF/YDEF info from .ctl
x0, dx = -52.7866058, 0.0360036008  # degrees
y0, dy = 22.7628231, 0.0360036008   # degrees

# Convert to km
lat_center = y0 + dy * ny / 2
km_per_deg_lat = 111.0
km_per_deg_lon = 111.0 * np.cos(np.deg2rad(lat_center))
x_km = (np.arange(nx) * dx) * km_per_deg_lon
y_km = (np.arange(ny) * dy) * km_per_deg_lat

# --- Read raw data ---
raw = np.fromfile(dat_file, dtype=">f4")
expected_size = nv * nz * ny * nx
if raw.size > expected_size:
    raw = raw[:expected_size]
data = raw.reshape((nv, nz, ny, nx))

# --- Extract time ---
time = get_time_from_ctl(ctl_file)
time_str = time.strftime("%Y-%m-%d %H:%M UTC") if time else "Unknown time"

# --- Set up figure ---
fig, ax = plt.subplots(figsize=(7,6))

# Initial frame
Z0 = data[R_index, 0, :, :]
Z0 = np.ma.masked_where(
    (Z0 < 0) | (Z0 > 80) | (Z0 == undef_val) | np.isnan(Z0),
    Z0
)
im = ax.imshow(Z0, origin="lower", cmap="turbo", vmin=0, vmax=50,
               extent=[x_km[0], x_km[-1], y_km[0], y_km[-1]])
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Reflectivity (dBZ)")
title = ax.set_title(f"Radar Reflectivity (lowest level)\n{time_str}")

ax.set_xlabel("X (km)")
ax.set_ylabel("Y (km)")

# --- Update function ---
def update(frame):
    Z0 = data[R_index, frame, :, :]
    Z0 = np.ma.masked_where(
        (Z0 < 0) | (Z0 > 80) | (Z0 == undef_val) | np.isnan(Z0),
        Z0
    )
    im.set_data(Z0)
    title.set_text(f"Radar Reflectivity (lowest level)\n{time_str} - Frame {frame+1}")
    return [im, title]

# --- Animate ---
num_frames = nz  # using vertical levels as time steps here
anim = FuncAnimation(fig, update, frames=num_frames, interval=500, blit=True)

# --- Save as MP4 ---
writer = FFMpegWriter(fps=2)
anim.save("radar_animation_single_flight_km_axes.mp4", writer=writer)
plt.close(fig)
print("Animation saved as radar_animation_single_flight_km_axes.mp4")
